# 04 - Pohon Keputusan
Materi Modul 4: pengantar pohon keputusan, membangun pohon, pemisahan berbasis entropi, kriteria pemisahan lainnya (Gini), pro dan kontra, lab.

Pohon keputusan memisahkan data berulang kali dengan aturan `fitur <= nilai`. Pemisahan dipilih yang paling **mengurangi ketidakmurnian (impurity)**.

- Entropi: $H = -\sum p_i \log_2 p_i$
- Gini: $G = 1 - \sum p_i^2$
- Information Gain = impurity induk - rata-rata tertimbang impurity anak

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

df = pd.read_csv('../datasets/breast_cancer.csv')
X, y = df.drop(columns='target'), df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

## 1. Hitung entropi dan Gini manual

In [ ]:
def entropi(label):
    p = np.bincount(label) / len(label)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

def gini(label):
    p = np.bincount(label) / len(label)
    return 1 - np.sum(p ** 2)

# Contoh: node dengan 10 sampel
for nama, label in [('Murni (10 vs 0)', np.array([1]*10)),
                    ('Campur (7 vs 3)', np.array([1]*7 + [0]*3)),
                    ('Paling campur (5 vs 5)', np.array([1]*5 + [0]*5))]:
    print(f'{nama:24s} | entropi={entropi(label):.3f} | gini={gini(label):.3f}')

## 2. Information gain untuk satu pemisahan
Contoh: pisahkan berdasarkan `worst perimeter <= 105`.

In [ ]:
kolom, batas = 'worst perimeter', 105
kiri = y_train[X_train[kolom] <= batas].values
kanan = y_train[X_train[kolom] > batas].values
induk = y_train.values

h_induk = entropi(induk)
h_anak = len(kiri) / len(induk) * entropi(kiri) + len(kanan) / len(induk) * entropi(kanan)

print('Entropi induk       :', round(h_induk, 4))
print('Entropi anak (rata2):', round(h_anak, 4))
print('Information gain    :', round(h_induk - h_anak, 4))

## 3. Bangun pohon

In [ ]:
tree = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print('Akurasi train:', round(tree.score(X_train, y_train), 4))
print('Akurasi test :', round(tree.score(X_test, y_test), 4))

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(tree, feature_names=X.columns, class_names=['jinak', 'ganas'], filled=True, fontsize=9)
plt.show()

In [ ]:
print(export_text(tree, feature_names=list(X.columns)))

## 4. Entropi vs Gini

In [ ]:
for kriteria in ['entropy', 'gini']:
    m = DecisionTreeClassifier(criterion=kriteria, max_depth=4, random_state=42)
    skor = cross_val_score(m, X_train, y_train, cv=5).mean()
    print(f'{kriteria:8s} | akurasi CV = {skor:.4f}')

## 5. Overfitting: pengaruh `max_depth`

In [ ]:
depth_list = range(1, 16)
acc_tr, acc_te = [], []
for d in depth_list:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    acc_tr.append(m.score(X_train, y_train))
    acc_te.append(m.score(X_test, y_test))

plt.plot(depth_list, acc_tr, marker='o', label='Train')
plt.plot(depth_list, acc_te, marker='o', label='Test')
plt.xlabel('max_depth'); plt.ylabel('Akurasi'); plt.legend()
plt.title('Pohon dalam = overfit'); plt.show()

## 6. Pengendalian kompleksitas (pruning)

In [ ]:
for params in [{}, {'max_depth': 4}, {'min_samples_leaf': 10}, {'ccp_alpha': 0.01}]:
    m = DecisionTreeClassifier(random_state=42, **params).fit(X_train, y_train)
    print(f'{str(params):26s} | daun={m.get_n_leaves():2d} | train={m.score(X_train, y_train):.3f} | test={m.score(X_test, y_test):.3f}')

## 7. Feature importance

In [ ]:
imp = pd.Series(DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train).feature_importances_, index=X.columns)
imp.sort_values().tail(8).plot(kind='barh')
plt.title('Fitur terpenting menurut pohon'); plt.show()

## Pro dan kontra
| Pro | Kontra |
|---|---|
| Mudah diinterpretasi dan divisualisasi | Mudah overfit |
| Tidak perlu scaling | Tidak stabil (data sedikit berubah -> pohon berubah) |
| Bisa fitur numerik dan kategorikal | Batas keputusan kaku (tegak lurus sumbu) |

## Catatan
- (cara memilih pemisahan, beda entropi dan gini, cara mengatasi overfit, dll.)